# DigiBot — Fine-Tuning Qwen3 1.7B → GGUF

### ANTES DE COMEÇAR:
1. **Ambiente de execução → Alterar tipo → GPU T4 → Salvar**
2. Execute as células **uma por uma**, na ordem.

---
### ORDEM DE EXECUÇÃO:
| Célula | O que faz | Tempo |
|--------|-----------|-------|
| 1 | Verifica GPU | 5s |
| 2 | **Instala dependências** (não reinicia) | 3 min |
| 3 | Upload do dataset | 1 min |
| 4 | Configurações | 5s |
| 5 | Carrega modelo Qwen3-1.7B | 5-10 min |
| 6 | Aplica LoRA | 30s |
| 7 | Prepara dataset | 1 min |
| 8 | **TREINA** | 15-25 min |
| 9 | Salva + backup no Drive | 2 min |
| 10 | Testa o modelo | 1 min |
| 11 | Merge LoRA + base | 5 min |
| 12 | Converte para GGUF Q4_K_M | 10 min |
| 13 | Baixa o GGUF | - |

---
## CÉLULA 1 — Verificar GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
    print('✅ GPU detectada! Pode continuar.')
else:
    print('❌ GPU NÃO encontrada!')
    print('Vá em: Ambiente de execução → Alterar tipo → GPU T4 → Salvar')

---
## CÉLULA 2 — Instalar dependências

⚠️ Esta célula **NÃO reinicia o kernel**. Após terminar, continue para a célula 3.

In [ ]:
import subprocess, sys

# Remover pacotes que causam conflito
print('Removendo pacotes conflitantes...')
for pkg in ['torchao', 'bitsandbytes']:
    subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', pkg],
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Instalar versões corretas
print('Instalando dependências (aguarde ~3 minutos)...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'bitsandbytes>=0.46.1',
    'transformers>=4.47.0',
    'peft>=0.13.0',
    'trl>=0.12.0',
    'accelerate>=0.35.0',
    'datasets>=3.0.0',
    'sentencepiece',
    'scipy',
])

# Verificar instalações
print('\nVerificando versões:')
checks = [
    ('bitsandbytes', 'bitsandbytes'),
    ('transformers', 'transformers'),
    ('peft', 'peft'),
]
ok = True
for name, mod in checks:
    r = subprocess.run([sys.executable, '-c', f'import {mod}; print({mod}.__version__)'],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f'  ✅ {name}: {r.stdout.strip()}')
    else:
        print(f'  ❌ {name}: FALHOU')
        ok = False

if ok:
    print('\n✅ Tudo instalado! Continue para a CÉLULA 3.')
else:
    print('\n❌ Alguma instalação falhou. Tente executar esta célula novamente.')

---
## CÉLULA 3 — Upload do dataset

Clique em ▶ e selecione o arquivo `digibot_dataset.jsonl`

In [ ]:
from google.colab import files
import os, json

print('Selecione o arquivo digibot_dataset.jsonl')
uploaded = files.upload()

if os.path.exists('digibot_dataset.jsonl'):
    with open('digibot_dataset.jsonl', encoding='utf-8') as f:
        lines = [l for l in f if l.strip()]
    print(f'\n✅ Dataset: {len(lines)} exemplos')
    ex = json.loads(lines[0])
    print(f'  Exemplo: {ex["conversations"][1]["content"][:60]}')
else:
    print('❌ Arquivo não encontrado.')

---
## CÉLULA 4 — Configurações

In [ ]:
BASE_MODEL  = 'Qwen/Qwen3-1.7B'
OUTPUT_DIR  = '/content/digibot-finetuned'
MERGED_DIR  = '/content/digibot-merged'

NUM_EPOCHS     = 3
BATCH_SIZE     = 2
GRAD_ACCUM     = 4
LEARNING_RATE  = 2e-4
MAX_SEQ_LENGTH = 512
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROP      = 0.05

print(f'✅ Configurações OK')
print(f'   Modelo: {BASE_MODEL}')
print(f'   Épocas: {NUM_EPOCHS} | Batch efetivo: {BATCH_SIZE * GRAD_ACCUM}')

---
## CÉLULA 5 — Carregar modelo base

⏱️ 5-10 minutos para baixar o Qwen3-1.7B (~3GB).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import bitsandbytes as bnb
print(f'bitsandbytes: {bnb.__version__}')

print(f'\nCarregando tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, trust_remote_code=True, use_fast=False,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
print('✅ Tokenizer OK')

print(f'\nCarregando modelo em 4-bit...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
print(f'\n✅ Modelo carregado!')
print(f'   Parâmetros: {model.num_parameters():,}')
print(f'   Memória GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB')

---
## CÉLULA 6 — Aplicar LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=LORA_DROP, bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('\n✅ LoRA aplicado!')

---
## CÉLULA 7 — Preparar dataset

In [ ]:
import json
from datasets import Dataset

raw_data = []
with open('digibot_dataset.jsonl', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            raw_data.append(json.loads(line))

def format_conversation(example):
    text = ''
    for msg in example['conversations']:
        role, content = msg['role'], msg['content']
        text += f'<|im_start|>{role}\n{content}<|im_end|>\n'
    return {'text': text}

def tokenize(example):
    result = tokenizer(example['text'], truncation=True,
                       max_length=MAX_SEQ_LENGTH, padding='max_length')
    result['labels'] = result['input_ids'].copy()
    return result

dataset   = Dataset.from_list(raw_data).map(format_conversation)
tokenized = dataset.map(tokenize, batched=False, remove_columns=['conversations','text'])
split     = tokenized.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

print(f'✅ Dataset pronto — Treino: {len(train_dataset)} | Validação: {len(eval_dataset)}')

---
## CÉLULA 8 — TREINAR

⏱️ 15-25 minutos. Acompanhe o `loss` diminuindo.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=50,
    lr_scheduler_type='cosine',
    fp16=True,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none',
    dataloader_num_workers=0,
    remove_unused_columns=False,
)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, padding=True, pad_to_multiple_of=8,
)
trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=eval_dataset,
    data_collator=data_collator,
)
print(f'🚀 Treinando {len(train_dataset)} exemplos por {NUM_EPOCHS} épocas...')
trainer.train()
print('\n✅ TREINAMENTO CONCLUÍDO!')

---
## CÉLULA 9 — Salvar + backup no Drive

Salva os adaptadores e faz backup no Google Drive.

In [ ]:
import os, shutil

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'✅ Salvo em: {OUTPUT_DIR}')
print(f'   Arquivos: {os.listdir(OUTPUT_DIR)}')

print('\nFazendo backup no Google Drive...')
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    dest = '/content/drive/MyDrive/digibot-finetuned'
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(OUTPUT_DIR, dest)
    print(f'✅ Backup em: {dest}')
except Exception as e:
    print(f'⚠️  Drive falhou: {e} — arquivos estão em disco.')

---
## CÉLULA 10 — Testar o modelo

In [ ]:
import torch

SYSTEM = ('Você é o DigiBot, assistente virtual da prefeitura municipal. '
          'Responda em português, de forma clara e objetiva.')

def perguntar(pergunta):
    prompt = (f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
              f'<|im_start|>user\n{pergunta}<|im_end|>\n'
              f'<|im_start|>assistant\n')
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7,
                                  top_p=0.9, do_sample=True,
                                  pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:],
                            skip_special_tokens=True).strip()

for q in ['Como solicito um serviço?', 'Como agendar consulta médica?']:
    print(f'👤 {q}')
    print(f'🤖 {perguntar(q)}\n')

---
## CÉLULA 11 — Merge LoRA + modelo base

⏱️ ~5 minutos. **Se a sessão tiver reiniciado, esta célula restaura o modelo do Drive automaticamente.**

In [ ]:
import subprocess, sys, os, torch

# Variáveis — definidas aqui para funcionar mesmo após reinício de sessão
BASE_MODEL = 'Qwen/Qwen3-1.7B'
OUTPUT_DIR = '/content/digibot-finetuned'
MERGED_DIR = '/content/digibot-merged'

# Restaurar do Drive se necessário
if not os.path.exists(OUTPUT_DIR) or not os.listdir(OUTPUT_DIR):
    print('Restaurando adaptadores do Google Drive...')
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    src = '/content/drive/MyDrive/digibot-finetuned'
    if not os.path.exists(src):
        raise FileNotFoundError(f'Backup não encontrado em {src}. Execute as células 8 e 9 primeiro.')
    if os.path.exists(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)
    shutil.copytree(src, OUTPUT_DIR)
    print(f'✅ Restaurado: {os.listdir(OUTPUT_DIR)}')
else:
    print(f'✅ Adaptadores em disco: {os.listdir(OUTPUT_DIR)}')

# Remover torchao — conflita com PeftModel em CPU
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for mod in list(sys.modules.keys()):
    if 'torchao' in mod:
        del sys.modules[mod]

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print('\nCarregando tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, use_fast=False)

print('Carregando modelo base em float16 (CPU, ~5 min)...')
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=torch.float16, device_map='cpu', trust_remote_code=True,
)

print('Aplicando e mesclando LoRA...')
merged = PeftModel.from_pretrained(base, OUTPUT_DIR).merge_and_unload()

print(f'Salvando em {MERGED_DIR}...')
os.makedirs(MERGED_DIR, exist_ok=True)
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

size = sum(os.path.getsize(os.path.join(MERGED_DIR, f))
           for f in os.listdir(MERGED_DIR)
           if os.path.isfile(os.path.join(MERGED_DIR, f)))
print(f'\n✅ Merge concluído! {size/1e9:.2f} GB em {MERGED_DIR}')

---
## CÉLULA 12 — Converter para GGUF Q4_K_M

⏱️ ~10 minutos.

In [ ]:
import subprocess, sys, os

MERGED_DIR = '/content/digibot-merged'
GGUF_F16   = '/content/digibot-f16.gguf'
GGUF_Q4    = '/content/digibot-q4_k_m.gguf'

if not os.path.exists(MERGED_DIR) or not os.listdir(MERGED_DIR):
    raise RuntimeError('MERGED_DIR vazio. Execute a célula 11 primeiro.')

# Clonar llama.cpp
if not os.path.exists('llama.cpp'):
    print('Clonando llama.cpp...')
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/ggerganov/llama.cpp', 'llama.cpp'], check=True)

# Instalar dependências Python
print('Instalando dependências...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'gguf', 'sentencepiece', 'transformers', 'numpy',
])

# Encontrar script de conversão
convert_script = None
for c in ['llama.cpp/convert_hf_to_gguf.py', 'llama.cpp/convert-hf-to-gguf.py', 'llama.cpp/convert.py']:
    if os.path.exists(c):
        convert_script = c
        break
print(f'Script de conversão: {convert_script}')

# Converter para F16
if not os.path.exists(GGUF_F16):
    print('\nConvertendo para GGUF F16...')
    r = subprocess.run([sys.executable, convert_script, MERGED_DIR,
                        '--outfile', GGUF_F16, '--outtype', 'f16'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print('ERRO:\n', r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError('Conversão F16 falhou.')
    print(f'✅ F16: {os.path.getsize(GGUF_F16)/1e9:.2f} GB')
else:
    print(f'✅ F16 já existe: {os.path.getsize(GGUF_F16)/1e9:.2f} GB')

# Instalar llama-cpp-python para quantização (sem precisar compilar)
print('\nInstalando llama-cpp-python (quantizador)...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'llama-cpp-python',
], env={**os.environ, 'CMAKE_ARGS': '-DGGML_CUDA=OFF'})

# Quantizar usando script Python do llama.cpp
print('Quantizando para Q4_K_M...')

# Tentar usar o binário compilado se existir
quantize_bin = None
for b in ['llama.cpp/llama-quantize', 'llama.cpp/build/bin/llama-quantize']:
    if os.path.exists(b):
        quantize_bin = b
        break

if quantize_bin:
    r = subprocess.run([quantize_bin, GGUF_F16, GGUF_Q4, 'Q4_K_M'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print('ERRO quantize binário:\n', r.stderr[-1000:])
        quantize_bin = None

if not quantize_bin:
    # Compilar com cmake (mais confiável que make no Colab)
    print('Compilando com cmake...')
    subprocess.run(['apt-get', 'install', '-y', '-q', 'cmake'], check=True,
                   capture_output=True)
    build_dir = 'llama.cpp/build'
    os.makedirs(build_dir, exist_ok=True)
    subprocess.run(['cmake', '..', '-DGGML_CUDA=OFF', '-DCMAKE_BUILD_TYPE=Release'],
                   cwd=build_dir, check=True, capture_output=True)
    subprocess.run(['cmake', '--build', '.', '--target', 'llama-quantize', '-j4'],
                   cwd=build_dir, check=True)
    r = subprocess.run(['llama.cpp/build/bin/llama-quantize', GGUF_F16, GGUF_Q4, 'Q4_K_M'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print('ERRO:\n', r.stderr[-1000:])
        raise RuntimeError('Quantização falhou.')

size_q4  = os.path.getsize(GGUF_Q4) / 1e9
size_f16 = os.path.getsize(GGUF_F16) / 1e9
print(f'\n✅ GGUF Q4_K_M gerado!')
print(f'   {GGUF_Q4}')
print(f'   {size_q4:.2f} GB  (era {size_f16:.2f} GB — redução de {(1-size_q4/size_f16)*100:.0f}%)')

---
## CÉLULA 13 — Baixar o GGUF

In [ ]:
from google.colab import files
import os

GGUF_Q4 = '/content/digibot-q4_k_m.gguf'
print(f'Tamanho: {os.path.getsize(GGUF_Q4)/1e9:.2f} GB')
print('Iniciando download...')
files.download(GGUF_Q4)

---
## CÉLULA 14 — (ALTERNATIVA) Salvar GGUF no Drive

In [ ]:
# DESCOMENTE PARA USAR
# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
# shutil.copy2('/content/digibot-q4_k_m.gguf', '/content/drive/MyDrive/digibot-q4_k_m.gguf')
# print('✅ Salvo no Drive!')
print('Descomente para usar.')

---
## CÉLULA 15 — Como usar no Docker

In [ ]:
print('''
=== COMO USAR NO SERVIDOR ===

1. Copie para o VPS:
   scp digibot-q4_k_m.gguf usuario@seu-vps:/caminho/

2. Coloque no container:
   docker cp digibot-q4_k_m.gguf digiurban-llamacpp:/models/

3. No docker-compose.vps.yml troque:
   ggml-org/Qwen3-1.7B-GGUF:Q4_K_M
   por:
   /models/digibot-q4_k_m.gguf

4. Reinicie:
   docker compose -f docker-compose.vps.yml restart llamacpp
''')